# Sketchy Parking Lot EDA

In [2]:
# Import and setup
import json
import os
import pandas as pd

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Domain parsing
import tldextract

## Import Data and Preparation

In [4]:
'''
recursively load data
'''

data_location = '../data/parked_suspicious/'

# Load summary
with open(f"{data_location}_summary.json") as f:
    summary = json.load(f)

dfs = []

# Iterate over summary labels to combine all JSONL files
for label in summary.keys():
    clean_label = label.replace(':', '-')

    filename = f"{data_location + clean_label}.jsonl"
    
    if not os.path.exists(filename):
        print(f"Missing: {filename}")
        continue
    
    try:
        df = pd.read_json(filename, lines=True)
        dfs.append(df)
        print(f"Loaded {filename} ({len(df)} rows)")
        
    except ValueError as e:
        print(f"Error reading {filename}: {e}")

# Combine all JSONL files into single DataFrame
combined_df = pd.concat(dfs, ignore_index=True)

print("Total rows:", len(combined_df))
print("Services:", combined_df["service"].value_counts())

Loaded ../data/parked_suspicious/other-unclassified.jsonl (1000 rows)
Loaded ../data/parked_suspicious/other-empty-title.jsonl (1000 rows)
Loaded ../data/parked_suspicious/default-server.jsonl (1000 rows)
Loaded ../data/parked_suspicious/sedo.jsonl (1000 rows)
Loaded ../data/parked_suspicious/searchhounds.jsonl (1000 rows)
Loaded ../data/parked_suspicious/wix-unconnected.jsonl (1000 rows)
Loaded ../data/parked_suspicious/hugedomains.jsonl (1000 rows)
Loaded ../data/parked_suspicious/godaddy.jsonl (1000 rows)
Loaded ../data/parked_suspicious/afternic.jsonl (1000 rows)
Loaded ../data/parked_suspicious/expired-domain.jsonl (1000 rows)
Loaded ../data/parked_suspicious/atom.jsonl (1000 rows)
Loaded ../data/parked_suspicious/spaceship.jsonl (1000 rows)
Loaded ../data/parked_suspicious/other-domain-as-title.jsonl (642 rows)
Loaded ../data/parked_suspicious/dynadot.jsonl (619 rows)
Loaded ../data/parked_suspicious/brandbucket.jsonl (440 rows)
Loaded ../data/parked_suspicious/default-nginx.json

In [5]:
combined_df.head()

,url,domain,service,hops,redirectChain,finalUrl,title,textSnippet
0,http://aaa-fuerteventura-appartments.com,aaa-fuerteventura-appartments.com,other:unclassified,5,[{'from': 'https://aaa-fuerteventura-appartmen...,https://welcome.mywebsite-editor.com/render/ww...,Bald verfügbar,Bald verfügbar
1,http://a-afolienfahrzeuge.at,a-afolienfahrzeuge.at,other:unclassified,3,"[{'from': 'https://a-afolienfahrzeuge.at', 'to...",https://www.a-afolienfahrzeuge.at/,Connect Your Domain,Connect Your Domain You're almost done! Your c...
2,http://a-lola-estilo.com,a-lola-estilo.com,other:unclassified,3,"[{'from': 'https://a-lola-estilo.com', 'to': '...",https://www.a-lola-estilo.com/password,A Lola Estilo – Abertura em breve,A Lola Estilo – Abertura em breve A Lola Estil...
3,http://a-petrenko.com,a-petrenko.com,other:unclassified,3,"[{'from': 'https://a-petrenko.com', 'to': 'htt...",https://a-petrenko.com/info/,a-petrenko,a-petrenko
4,http://a-point-of-view.de,a-point-of-view.de,other:unclassified,3,"[{'from': 'https://a-point-of-view.de', 'to': ...",https://p2m-1.forsaledomain.net/true2.php?id=a...,Just a moment...,Just a moment... p2m-1.forsaledomain.net Verif...


In [6]:
combined_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15673 entries, 0 to 15672
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   url            15673 non-null  object
 1   domain         15673 non-null  object
 2   service        15673 non-null  object
 3   hops           15673 non-null  int64 
 4   redirectChain  15673 non-null  object
 5   finalUrl       15673 non-null  object
 6   title          15673 non-null  object
 7   textSnippet    15673 non-null  object
dtypes: int64(1), object(7)
memory usage: 979.7+ KB


In [7]:
combined_df.describe()

,hops
count,15673.000000
mean,0.710202
std,0.998587
min,0.000000
25%,0.000000
50%,0.000000
75%,2.000000
max,5.000000


## Text Analysis for Identification of Parked or Ad Domains 

## Data Wrangling

Feature engineer

In [ ]:
'''
extract TLD, domain and subdomain using tldextract

Parse the full domain for analysis.
'''
# change domain to domain_full to preserve original domain for tldextract
combined_df["domain_full"] = combined_df["domain"]
combined_df.drop("domain", axis=1, inplace=True)

def extract_domain_parts(domain):
    ext = tldextract.extract(domain)
    return pd.Series({
        "subdomain": ext.subdomain,
        "domain_name": ext.domain,
        "tld": ext.suffix
    })

combined_df[["subdomain", "domain_name", "tld"]] = combined_df["domain_full"].apply(extract_domain_parts)

In [ ]:
'''
domain length feature

Identify how many characters the domain has.
'''
combined_df["len_domain_full"] = combined_df["domain_full"].apply(len)

In [ ]:
'''
is_suspicious feature

This feature is derived from the amount of hops (redirects) domain has.
If a domain has more than 2 hops, we can consider it suspicious.
Subject to change based on analysis of hops distribution.
'''
threshold_hops = 2
combined_df["is_suspicious"] = combined_df["hops"].apply(lambda x: 1 if x > threshold_hops else 0)

In [ ]:
'''
Shannons entropy of the domain feature

Calculate the Shannon entropy of the domain name to measure its randomness.
Higher entropy may indicate a more complex and potentially suspicious domain.
'''


In [8]:
combined_df.head()

,url,domain,service,hops,redirectChain,finalUrl,title,textSnippet
0,http://aaa-fuerteventura-appartments.com,aaa-fuerteventura-appartments.com,other:unclassified,5,[{'from': 'https://aaa-fuerteventura-appartmen...,https://welcome.mywebsite-editor.com/render/ww...,Bald verfügbar,Bald verfügbar
1,http://a-afolienfahrzeuge.at,a-afolienfahrzeuge.at,other:unclassified,3,"[{'from': 'https://a-afolienfahrzeuge.at', 'to...",https://www.a-afolienfahrzeuge.at/,Connect Your Domain,Connect Your Domain You're almost done! Your c...
2,http://a-lola-estilo.com,a-lola-estilo.com,other:unclassified,3,"[{'from': 'https://a-lola-estilo.com', 'to': '...",https://www.a-lola-estilo.com/password,A Lola Estilo – Abertura em breve,A Lola Estilo – Abertura em breve A Lola Estil...
3,http://a-petrenko.com,a-petrenko.com,other:unclassified,3,"[{'from': 'https://a-petrenko.com', 'to': 'htt...",https://a-petrenko.com/info/,a-petrenko,a-petrenko
4,http://a-point-of-view.de,a-point-of-view.de,other:unclassified,3,"[{'from': 'https://a-point-of-view.de', 'to': ...",https://p2m-1.forsaledomain.net/true2.php?id=a...,Just a moment...,Just a moment... p2m-1.forsaledomain.net Verif...


## Classify Based on Engineered Features

Classify as parked, expired, default, or active domain.

## Data Distributions and Visualizations

In [18]:
combined_df.columns

Index(['url', 'service', 'hops', 'redirectChain', 'finalUrl', 'title',
       'textSnippet', 'domain_full', 'subdomain', 'domain_name', 'tld'],
      dtype='object')

## Save JSONL

In [19]:
# save as JSONL for future use

# combined_df.to_json('data/parked_suspicious/_combined_data.jsonl', orient='records', lines=True)